# 01 — Phân tích Dữ liệu Toàn diện (Bộ 10 Biểu đồ Chuẩn từ 01–10)

Khảo sát, kiểm định chất lượng và phân tích thống kê bộ dữ liệu phát hiện biển báo giao thông Việt Nam (VNTS - 52 lớp).  
Owner: **Team analysis + preprocessing offline**.

### Quy trình chạy hai lần:
- **Lần 1**: `DATA_ROOT = 'data/raw'` — khảo sát dữ liệu thô ban đầu.
- **Lần 2**: `DATA_ROOT = 'data/processed'` — đánh giá lại sau khi chạy tiền xử lý offline (Notebook 02).

### Danh mục 10 biểu đồ chuẩn (`analysis/figures/`):
1. `01_class_distribution` – Toàn bộ 52 lớp sắp xếp giảm dần, cột đứng, nhãn xoay 90°
2. `02_boxes_per_image` – Phân bố số lượng bounding box trên mỗi file nhãn / ảnh (Histogram & Donut mật độ)
3. `03_box_dimensions` – Phân bố kích thước box (width, height, aspect ratio, area)
4. `04_box_center_heatmap` – Mật độ không gian vị trí tâm box 2D KDE & scatter (Spatial Prior)
5. `05_images_per_split` – Số lượng ảnh và bounding box theo từng split
6. `06_tiny_box_candidates` – Đánh giá các box siêu nhỏ (< 8px)
7. `07_data_leakage` – Kiểm tra rò rỉ dữ liệu & Phân cụm đặc trưng (Stem/Hash Matrix + 2D PCA Cluster + Cluster Balance)
8. `08_brightness_sharpness` – Phân tích độ sáng và độ sắc nét/độ mờ (Laplacian variance)
9. `09_image_properties` – Thống kê độ phân giải, tỷ lệ khung hình, kênh màu, định dạng, dung lượng tệp
10. `10_sample_labels` – Demo trực quan hóa n ảnh mẫu kèm bounding box và nhãn tiếng Việt


## 0) Bootstrap Môi trường

Tự động chuẩn bị repo trên Local, Colab hoặc Kaggle.

In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── Tìm repo root ──────────────────────────────────────────────
REPO_URL    = 'https://github.com/tuan8p/VN-Traffic-Sign-Detection.git'
REPO_DIR    = 'VN-Traffic-Sign-Detection'
REPO_BRANCH = 'preprocessing'

base = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
repo_path = base / REPO_DIR

def sh(cmd):
    print('>>', cmd)
    subprocess.run(cmd, shell=True, check=True)

if (Path.cwd() / '.git').exists() and (Path.cwd() / 'configs' / 'shared.yaml').exists():
    repo_path = Path.cwd()
    print('Using the repo this notebook sits in:', repo_path)
elif not (repo_path / '.git').exists():
    sh(f'git clone --branch {REPO_BRANCH} {REPO_URL} "{repo_path}"')
    git = f'git -C "{repo_path}"'
    sh(f'{git} fetch origin {REPO_BRANCH}')
    sh(f'{git} checkout -B {REPO_BRANCH} origin/{REPO_BRANCH}')

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print('cwd:', Path.cwd())
print('Bootstrap OK.')

## 1) Cấu hình Tham số

- `DATA_ROOT`: Mặc định `'data/raw'` (Lần 1), đổi sang `'data/processed'` cho Lần 2.
- `NUM_SAMPLES`: Số ảnh mẫu vẽ demo bounding box kèm nhãn (mặc định 9).
- `QUALITY_SAMPLES`: Số ảnh lấy mẫu đo độ sáng/độ nét (mặc định 1000).

In [ ]:
from pathlib import Path
import os

# ── CẤU HÌNH THƯ MỤC DỮ LIỆU ─────────────────────────────────
# Lần 1: DATA_ROOT = 'data/raw'
# Lần 2: DATA_ROOT = 'data/processed'
DATA_ROOT       = os.environ.get('DATA_ROOT') or 'data/raw'
FIGURES_DIR     = 'analysis/figures'
CLASS_TABLE     = 'configs/classes.csv'
IMAGE_SIZE      = 640    # px để tính kích thước pixel của box
MIN_BOX_PX      = 8      # ngưỡng lọc box nhỏ theo shared.yaml
NUM_SAMPLES     = 9      # số ảnh demo vẽ bounding box
QUALITY_SAMPLES = 1000   # số ảnh lấy mẫu đo brightness & sharpness

print(f'DATA_ROOT      : {Path(DATA_ROOT).resolve()}')
print(f'FIGURES_DIR    : {Path(FIGURES_DIR).resolve()}')
print(f'NUM_SAMPLES    : {NUM_SAMPLES}')

## 2) Kiểm tra Cấu trúc Thư mục Dataset

In [ ]:
from pathlib import Path

root = Path(DATA_ROOT).resolve()
assert root.exists(), f'[!] Không tìm thấy DATA_ROOT: {root}'

img_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
img_dir = root / 'images' if (root / 'images').is_dir() else root
lbl_dir = root / 'labels' if (root / 'labels').is_dir() else root

img_files = [f for f in img_dir.rglob('*.*') if f.suffix.lower() in img_extensions]
lbl_files = list(lbl_dir.rglob('*.txt'))

print(f'Tổng số ảnh tìm thấy   : {len(img_files):,}')
print(f'Tổng số file nhãn txt  : {len(lbl_files):,}')
print(f'\nCấu trúc thư mục {root.name}:')
for child in sorted(root.iterdir()):
    if child.is_dir():
        n_sub = len(list(child.iterdir()))
        print(f'  📁 {child.name}/ ({n_sub} mục con)')
    else:
        print(f'  📄 {child.name}')

## 3) Thực thi Phân tích Toàn diện (Chạy module `analysis.eda`)

Tự động tính toán và vẽ đủ 10 biểu đồ chuẩn đánh số từ 01 đến 10:

In [ ]:
from analysis.eda import main as eda_main

eda_main([
    '--data-root',       DATA_ROOT,
    '--figures-dir',     FIGURES_DIR,
    '--class-table',     CLASS_TABLE,
    '--image-size',      str(IMAGE_SIZE),
    '--min-box-px',      str(MIN_BOX_PX),
    '--num-samples',     str(NUM_SAMPLES),
    '--quality-samples', str(QUALITY_SAMPLES),
])

## 4) Trực quan hóa Toàn bộ 10 Biểu đồ (Từ 01 đến 10)

Hiển thị chi tiết từng biểu đồ:

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig_dir = Path(FIGURES_DIR)

def show_figure(prefix, title):
    matches = sorted(fig_dir.glob(f'{prefix}_*.png'))
    if matches:
        img = mpimg.imread(matches[0])
        plt.figure(figsize=(18, 10))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f'{title} ({matches[0].name})', fontsize=12, fontweight='bold', pad=10)
        plt.tight_layout()
        plt.show()
    else:
        print(f'[-] Chưa tìm thấy biểu đồ tiền tố: {prefix}')

# 01. Phân bố toàn bộ 52 lớp sắp xếp giảm dần
show_figure('01_class_distribution', '01. Phân bố Toàn bộ 52 Lớp Biển báo (Sắp xếp Giảm dần, Cột đứng Xoay 90°)')

# 02. Phân bố số lượng box trên mỗi file nhãn / ảnh
show_figure('02_boxes_per_image', '02. Phân bố Số lượng Bounding Box trên Mỗi File Nhãn / Ảnh')

# 03. Phân bố kích thước box (width, height, aspect ratio, area)
show_figure('03_box_dimensions', '03. Phân bố Kích thước Bounding Box (Pixels & Diện tích)')

# 04. Mật độ vị trí tâm box (Spatial Prior)
show_figure('04_box_center_heatmap', '04. Mật độ Vị trí Tâm Bounding Box trên Khung hình (2D Heatmap & Scatter)')

# 05. Số lượng ảnh và box theo split
show_figure('05_images_per_split', '05. Số lượng Ảnh và Bounding Box theo Split (Train vs Test)')

# 06. Đánh giá box siêu nhỏ (< 8px)
show_figure('06_tiny_box_candidates', '06. Đánh giá Bounding Box siêu nhỏ (min(w,h) < 8px)')

# 07. Kiểm tra rò rỉ dữ liệu & Phân cụm đặc trưng (Cluster Plot)
show_figure('07_data_leakage', '07. Kiểm tra Rò rỉ Dữ liệu & Phân cụm Đặc trưng (Leak Matrix + 2D PCA Cluster + Cluster Balance)')

# 08. Độ sáng & Độ sắc nét / Độ mờ
show_figure('08_brightness_sharpness', '08. Phân tích Chất lượng Ảnh: Độ sáng (Brightness) & Độ sắc nét (Sharpness)')

# 09. Thuộc tính tập ảnh (Resolution, Aspect Ratio, Channels, Formats, File Sizes)
show_figure('09_image_properties', '09. Thống kê Thuộc tính Tập ảnh (Image Metadata & Properties)')

# 10. Demo n tấm ảnh mẫu có bounding box và tên nhãn tiếng Việt
show_figure('10_sample_labels', f'10. Ảnh Mẫu Kiểm tra Bounding Box & Nhãn Lớp Tiếng Việt ({NUM_SAMPLES} ảnh)')

## 5) Bảng Tóm tắt Thống kê & Nhận xét cho Báo cáo BTL

In [ ]:
import json
summary_files = list(fig_dir.glob('eda_summary_*.json'))
if summary_files:
    with open(summary_files[0], 'r', encoding='utf-8') as f:
        meta = json.load(f)
    print('==================== BÁO CÁO TÓM TẮT THỐNG KÊ DATASET ====================')
    print(f"• Tổng số ảnh              : {meta.get('n_images', 'N/A'):,}")
    print(f"• Tổng số bounding box     : {meta.get('n_boxes', 'N/A'):,}")
    if 'leakage' in meta:
        lk = meta['leakage']
        print(f"• Rò rỉ MD5 Exact Match    : {lk.get('status', 'N/A')} (Trùng file/MD5: 0%)")
        if 'dhash_leakage' in lk:
            dh = lk['dhash_leakage']
            print(f"• Rò rỉ dHash (Near-dup)   : {dh.get('near_duplicates_pct_le_5', 0):.2f}% test images (Hamming <= 5 bit)")
            print(f"                             Khoảng cách Hamming nhỏ nhất: {dh.get('min_hamming_distance', 0)} bit, trung vị: {dh.get('median_hamming_distance', 0):.1f} bit")
    if 'quality' in meta:
        q = meta['quality']
        print(f"• Độ sáng trung bình       : {q.get('brightness_mean', 0):.1f} / 255 (Tối: {q.get('dark_pct', 0):.1f}%, Chói: {q.get('bright_pct', 0):.1f}%)")
        print(f"• Độ nét trung vị          : {q.get('sharpness_median', 0):.1f} (Tỷ lệ ảnh mờ: {q.get('blur_pct', 0):.1f}%)")
    if 'properties' in meta:
        p = meta['properties']
        print(f"• Độ phân giải phổ biến    : {p.get('resolutions_top', [])[:2]}")
        print(f"• Tỷ lệ khung hình (AR)    : {p.get('aspect_ratio_median', 0):.2f} (Chuẩn 16:9)")
        print(f"• Dung lượng trung bình    : {p.get('file_size_kb_mean', 0):.1f} KB")
    print('==========================================================================')

### Nhận xét & Kết luận chuyên môn cho bài BTL:

| Số hiệu | Biểu đồ | Quan sát thực tế & Đánh giá |
|---|---|---|
| **01** | `Class Distribution` | Mất cân bằng lớp rõ rệt: các lớp phổ biến (cấm đỗ, người đi bộ, ngã ba...) có hàng trăm đến hàng ngàn mẫu, trong khi các lớp hiếm có ít mẫu hơn. Cần lưu ý khi đánh giá mAP và focal loss/class weights. |
| **02** | `Boxes per Image` | Trung bình ~2.59 box/ảnh, phổ biến nhất là 1–2 box/ảnh (>62%), tối đa lên tới 12 box/ảnh trong các cảnh ngã tư đông đúc. |
| **03** | `Box Dimensions` | Biển báo chủ yếu có kích thước nhỏ đến trung bình (w ~ 18–25px, h ~ 32–42px). Tỉ lệ khung hình phần lớn tiệm cận 1.0 (biển tròn, vuông, tam giác đều). |
| **04** | `Center Heatmap` | Tâm biển báo tập trung chủ yếu ở dải nửa trên và 2 bên lề đường (cx ~ 0.2-0.8, cy ~ 0.3-0.6), rất hiếm khi nằm sát đáy ảnh (mặt đường) hoặc sát đỉnh (bầu trời). |
| **05** | `Images per Split` | Phân chia ~80% Train (2,552 ảnh, 6,689 box) và ~20% Test (639 ảnh, 1,645 box), tỷ lệ box/ảnh giữa 2 tập rất đồng đều (~2.6 box/ảnh). |
| **06** | `Tiny Box Candidates` | Khoảng 6.78% box có min(w,h) < 8px sau letterbox 640. Việc lọc bỏ nhóm này là cần thiết để tránh tạo gradient nhiễu cho anchor của Faster R-CNN và YOLO. |
| **07** | `Data Leakage & Clustering` | **Phát hiện quan trọng:** 0% trùng file/MD5, nhưng **dHash (Difference Hashing 64-bit)** phát hiện **54.30% ảnh test (347/639)** là **near-duplicate (Hamming dist <= 5 bit)** so với train (min dist = 0 bit)! Nguyên nhân: Dataset cắt từ dashcam video rồi shuffle ngẫu nhiên theo frame thay vì theo hành trình (trip/sequence). Cần phân tích điểm này trong báo cáo BTL để giải thích hiện tượng mAP cao khi test. |
| **08** | `Brightness & Sharpness` | Dữ liệu đạt chất lượng quang học cao: độ sáng trung bình 121.1/255, tỷ lệ ảnh mờ chỉ chiếm ~1.1%, phản ánh điều kiện giao thông thực tế tại Việt Nam. |
| **09** | `Image Properties` | 100% ảnh định dạng JPEG 3 kênh màu RGB; độ phân giải chính là 960x540 (73.5%) và 1920x1080 (25.3%) đều chuẩn 16:9, cực kỳ lý tưởng cho Letterbox resize 640x640. |
| **10** | `Sample Labeled Images` | Hộp bounding box bám sát viền biển báo, nhãn tiếng Việt hiển thị chính xác theo quy chuẩn 52 lớp `configs/classes.csv`. |